# Can we predict room occupancy from sensor readings?

Data: `data.csv` - one row per ~minute of Temperature, Humidity, Light, CO2 and HumidityRatio, with a binary `Occupancy` label.

**Approach**
1. Inspect the data (balance, time coverage, gaps).
2. Split **chronologically by day**. This is time-series data, so a random row split would put neighbouring minutes in both train and test and overstate performance.
3. Compare simple baselines to logistic regression and tree ensembles, using day-grouped cross-validation on the training period and a final held-out test period.
4. Check which features matter and where the model fails.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             average_precision_score, ConfusionMatrixDisplay, RocCurveDisplay,
                             classification_report)

df = pd.read_csv('data.csv', parse_dates=['date'])
df['day'] = df.date.dt.date
print(df.shape, '| missing values:', int(df.isna().sum().sum()), '| duplicates:', int(df.duplicated().sum()))
print(df.date.min(), '->', df.date.max())
df.describe().T

## 1. Data overview

In [ ]:
gaps = df.date.diff().dt.total_seconds()
print('Gaps >2 min:', (gaps > 120).sum(), '| largest gap (h):', round(gaps.max()/3600, 1))
print(df.Occupancy.value_counts(normalize=True).rename('share').to_frame())
daily = df.groupby('day').Occupancy.agg(share_occupied='mean', rows='size')
daily['weekday'] = pd.to_datetime(daily.index).day_name()
daily

About 23% of minutes are occupied, so plain accuracy flatters: always predicting "empty" scores ~77%. I also report balanced accuracy, F1, ROC AUC and PR AUC. Weekend days are entirely empty, and some days are only partially recorded.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(13, 11), sharex=True)
for ax, c in zip(axes, ['Temperature','Humidity','Light','CO2','HumidityRatio']):
    ax.plot(df.date, df[c], lw=.5, color='tab:blue')
    ax2 = ax.twinx(); ax2.fill_between(df.date, df.Occupancy, step='mid', alpha=.2, color='tab:red'); ax2.set_yticks([])
    ax.set_ylabel(c)
axes[0].set_title('Sensor readings (red shading = occupied)')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, c in zip(axes, ['Temperature','Humidity','Light','CO2','HumidityRatio']):
    for o, col in [(0,'tab:blue'),(1,'tab:red')]:
        ax.hist(df.loc[df.Occupancy==o, c], bins=40, alpha=.5, density=True, color=col, label=f'occ={o}')
    ax.set_title(c)
axes[0].legend(); plt.tight_layout(); plt.show()
df[['Temperature','Humidity','Light','CO2','HumidityRatio','Occupancy']].corr()['Occupancy'].round(3)

## 2. Train / test split

Days are split chronologically: the first ~70% of days train, the remaining days test. The test period is entirely after the training data.

In [ ]:
days = sorted(df.day.unique())
n_train = int(len(days)*0.7)
train_days, test_days = days[:n_train], days[n_train:]
train, test = df[df.day.isin(train_days)], df[df.day.isin(test_days)]
print('train:', train_days[0], '->', train_days[-1], '| rows', len(train), '| occupied', round(train.Occupancy.mean(),3))
print('test: ', test_days[0], '->', test_days[-1], '| rows', len(test), '| occupied', round(test.Occupancy.mean(),3))

## 3. Models

Features: the five raw sensor readings (no time-of-day features, so the model has to rely on the physical signals). Models are compared with **GroupKFold by day** on the training period, then fit on all training data and scored once on the test period.

In [ ]:
FEATS = ['Temperature','Humidity','Light','CO2','HumidityRatio']
Xtr, ytr, gtr = train[FEATS], train.Occupancy, train.day
Xte, yte = test[FEATS], test.Occupancy

class LightThreshold(BaseEstimator, ClassifierMixin):
    """One-feature rule: occupied if Light > t, with t chosen on the training data (balanced accuracy)."""
    def fit(self, X, y):
        ts = np.quantile(X['Light'], np.linspace(.05, .95, 181))
        self.t_ = max(ts, key=lambda t: balanced_accuracy_score(y, X['Light'] > t))
        self.classes_ = np.array([0, 1]); return self
    def predict_proba(self, X):
        p = 1/(1+np.exp(-(X['Light']-self.t_)/20)); return np.c_[1-p, p]
    def predict(self, X): return (X['Light'] > self.t_).astype(int).values

models = {
    'majority class': DummyClassifier(strategy='prior'),
    'Light threshold': LightThreshold(),
    'logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'random forest': RandomForestClassifier(300, min_samples_leaf=5, random_state=0, n_jobs=-1),
    'gradient boosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200, random_state=0),
}
def metrics(y, pred, proba):
    return dict(accuracy=accuracy_score(y,pred), bal_acc=balanced_accuracy_score(y,pred), f1=f1_score(y,pred, zero_division=0),
                roc_auc=roc_auc_score(y,proba), pr_auc=average_precision_score(y,proba))

gkf = GroupKFold(n_splits=5)
rows = {}
for name, m in models.items():
    proba = cross_val_predict(m, Xtr, ytr, groups=gtr, cv=gkf, method='predict_proba')[:,1]
    rows[name] = metrics(ytr, (proba > .5).astype(int), proba)
print('Day-grouped CV on the training period')
pd.DataFrame(rows).T.round(4)

## 4. Held-out test period

In [ ]:
fitted, rows = {}, {}
for name, m in models.items():
    fitted[name] = clone(m).fit(Xtr, ytr)
    rows[name] = metrics(yte, fitted[name].predict(Xte), fitted[name].predict_proba(Xte)[:,1])
print('Light threshold t =', round(fitted['Light threshold'].t_, 1))
pd.DataFrame(rows).T.round(4)

In [ ]:
best = 'logistic regression'  # best on day-grouped CV (chosen before looking at test)
m = fitted[best]; proba = m.predict_proba(Xte)[:,1]; pred = m.predict(Xte)
print(classification_report(yte, pred, target_names=['empty','occupied'], digits=3))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=['empty','occupied'], ax=ax[0], colorbar=False)
ax[0].set_title(f'{best}: confusion matrix (test)')
for n in ['Light threshold','logistic regression','random forest','gradient boosting']:
    RocCurveDisplay.from_predictions(yte, fitted[n].predict_proba(Xte)[:,1], name=n, ax=ax[1])
ax[1].set_title('ROC (test)'); plt.tight_layout(); plt.show()

## 5. What drives the predictions, and where does it fail?

In [ ]:
pi = pd.Series(permutation_importance(m, Xte, yte, scoring='roc_auc', n_repeats=10, random_state=0).importances_mean, index=FEATS).sort_values()
pi.plot.barh(title='Permutation importance (drop in test ROC AUC)'); plt.tight_layout(); plt.show()

In [ ]:
t = test.assign(pred=pred, proba=proba, err=(pred != yte.values))
print(t.groupby('day').agg(rows=('err','size'), errors=('err','sum'), occ_share=('Occupancy','mean')))
run = t.Occupancy.ne(t.Occupancy.shift()).cumsum()
dist = np.minimum(t.groupby(run).cumcount(), t.groupby(run).cumcount(ascending=False))
print('\nShare of errors within 10 rows of a true state change:', round(t.err[dist < 10].sum()/max(t.err.sum(), 1), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(t.date, t.proba, lw=.6, label='P(occupied)')
ax.fill_between(t.date, t.Occupancy, step='mid', alpha=.25, color='tab:red', label='truth')
ax.scatter(t.date[t.err], t.proba[t.err], s=6, color='k', label='errors', zorder=3)
ax.legend(); ax.set_title('Test period: predicted probability vs truth'); plt.tight_layout(); plt.show()

## 6. Sensitivity to the split

With only ~17 days, one split can be lucky or unlucky. Here the logistic regression model is evaluated with day-grouped CV over **all** days, and I look at the spread across folds.

In [ ]:
Xall, yall, gall = df[FEATS], df.Occupancy, df.day
rows = []
for k, (tr, va) in enumerate(GroupKFold(n_splits=6).split(Xall, yall, gall)):
    mm = clone(models[best]).fit(Xall.iloc[tr], yall.iloc[tr])
    p = mm.predict_proba(Xall.iloc[va])[:,1]
    rows.append(dict(fold=k, days=gall.iloc[va].nunique(), **metrics(yall.iloc[va], (p > .5).astype(int), p)))
f = pd.DataFrame(rows).round(4); display(f)
f.drop(columns=['fold','days']).agg(['mean','std']).round(4)

## Conclusions

**Yes - occupancy is highly predictable from these sensors.**

- On the held-out final 6 days, logistic regression reaches ~99.2% accuracy, 0.994 balanced accuracy, F1 0.98 for the occupied class (recall 99.7%, precision 96.6%), ROC AUC ~0.997. The majority-class baseline gets 79.5% accuracy, so the lift is real.
- **Light does most of the work.** A single threshold on Light (~350 lux) already scores 99.2% accuracy / F1 0.98 on test, and 0.98 accuracy in CV. Logistic regression, random forest and gradient boosting are all within a fraction of a point of it, so the extra model complexity buys little. Treat the model as essentially "lights on => occupied" plus a small CO2/temperature correction.
- Most remaining errors (~2/3 for logistic regression) sit within a few minutes of an arrival or departure, when the sensors lag the true state.
- Day-grouped CV over all days gives mean accuracy 98.8% (std 0.8 pts), but F1 varies more across folds (0.82-0.99), since a fold with a weekend has few occupied rows.

**Caveats**
- Only ~17 days from one room in one fortnight, with 4 weekend days that are entirely empty. Generalisation to other rooms, seasons or lighting habits is untested; a light-based rule may break if lights are on in an empty room or occupants work with lights off.
- The Light threshold and scaling are learned from this room; a new deployment would need recalibration.
- The test result is one chronological split; the fold-to-fold spread in section 6 is a better guide to uncertainty than the single number.
- In the CV table, the majority-class ROC AUC (0.32) is below 0.5 only because a constant prior is pooled across folds with different base rates; ignore that cell.
